# Week 01: LangGraph Agent
### 726, Jamil

## 1. Dependencies

In [1]:
%pip install -q langgraph langchain-openai python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import os
import sqlite3
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

## 3. LLM setup

In [3]:
load_dotenv(Path('../../../.env'))
OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']

llm = ChatOpenAI(
    model='openai/gpt-4o-mini',
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)

# Vive check
response = llm.invoke('Write a one liner joke about ai takeover')
print(textwrap.fill(response.content, width=80))


Why did the AI break up with humanity? It found someone more compatible—like a
toaster!


## 4. Database

In-memory SQLite — 1v1 game leaderboard with three tables:
- `players` — current Elo, peak Elo, win/loss record
- `matches` — result, score, duration, mode
- `elo_history` — per-player Elo before/after every match

8 players, 26 matches (Oct–Nov 2025), 52 elo_history rows.

**Rivalry to notice:** zergling (1052 Elo) ranks above neonbyte (1048),
but neonbyte has won every single head-to-head between them — three times.

In [4]:
_conn = sqlite3.connect(':memory:', check_same_thread=False)

_conn.executescript('''
CREATE TABLE players (
    id        INTEGER PRIMARY KEY,
    username  TEXT NOT NULL UNIQUE,
    elo       INTEGER NOT NULL DEFAULT 1000,
    peak_elo  INTEGER NOT NULL DEFAULT 1000,
    wins      INTEGER NOT NULL DEFAULT 0,
    losses    INTEGER NOT NULL DEFAULT 0,
    joined_at TEXT NOT NULL
);

CREATE TABLE matches (
    id               INTEGER PRIMARY KEY,
    player1_id       INTEGER NOT NULL REFERENCES players(id),
    player2_id       INTEGER NOT NULL REFERENCES players(id),
    winner_id        INTEGER REFERENCES players(id),
    score_p1         INTEGER NOT NULL DEFAULT 0,
    score_p2         INTEGER NOT NULL DEFAULT 0,
    duration_seconds INTEGER NOT NULL,
    mode             TEXT NOT NULL CHECK(mode IN (\'ranked\',\'casual\',\'tournament\')),
    forfeit          INTEGER NOT NULL DEFAULT 0,
    played_at        TEXT NOT NULL
);

CREATE TABLE elo_history (
    id         INTEGER PRIMARY KEY,
    player_id  INTEGER NOT NULL REFERENCES players(id),
    match_id   INTEGER NOT NULL REFERENCES matches(id),
    elo_before INTEGER NOT NULL,
    elo_after  INTEGER NOT NULL,
    delta      INTEGER NOT NULL
);

-- Players
-- zergling (1052) > neonbyte (1048) in Elo, yet neonbyte won every H2H (M7, M12, M17)
INSERT INTO players VALUES
    (1, 'shadow_fox',  1120, 1120,  8, 0, '2025-10-01'),
    (2, 'zergling',    1052, 1052,  8, 4, '2025-10-01'),
    (3, 'neonbyte',    1048, 1048,  7, 4, '2025-10-01'),
    (4, 'grumpyking',   900, 1000,  0, 7, '2025-10-01'),
    (5, 'blueshift',   1028, 1044,  3, 1, '2025-10-01'),
    (6, 'pixelwitch',   940, 1000,  0, 4, '2025-10-01'),
    (7, 'ironclad',     944, 1000,  0, 4, '2025-10-01'),
    (8, 'rookie99',     968, 1000,  0, 2, '2025-10-01');

-- Matches (id, p1, p2, winner, score_p1, score_p2, duration_s, mode, forfeit, date)
INSERT INTO matches VALUES
    ( 1, 1, 8, 1, 5, 1, 1800, 'ranked',     0, '2025-10-01'),
    ( 2, 2, 4, 2, 5, 2, 2100, 'ranked',     0, '2025-10-03'),
    ( 3, 3, 6, 3, 5, 3, 2400, 'ranked',     0, '2025-10-05'),
    ( 4, 5, 7, 5, 5, 2, 1920, 'casual',     0, '2025-10-07'),
    ( 5, 1, 4, 1, 5, 1, 1680, 'ranked',     0, '2025-10-09'),
    ( 6, 2, 6, 2, 5, 2, 2040, 'ranked',     0, '2025-10-11'),
    ( 7, 3, 2, 3, 5, 4, 3120, 'ranked',     0, '2025-10-13'),  -- neonbyte beats zergling #1
    ( 8, 1, 3, 1, 5, 3, 2700, 'ranked',     0, '2025-10-15'),
    ( 9, 2, 7, 2, 5, 2, 1800, 'ranked',     0, '2025-10-17'),
    (10, 3, 4, 3, 5, 2, 2160, 'ranked',     0, '2025-10-19'),
    (11, 1, 2, 1, 5, 3, 2880, 'tournament', 0, '2025-10-21'),
    (12, 3, 2, 3, 5, 4, 3300, 'ranked',     0, '2025-10-23'),  -- neonbyte beats zergling #2
    (13, 2, 4, 2, 5, 1, 1560, 'ranked',     0, '2025-10-25'),
    (14, 1, 3, 1, 5, 2, 2400, 'ranked',     0, '2025-10-27'),
    (15, 5, 6, 5, 5, 3, 2280, 'casual',     0, '2025-10-29'),
    (16, 2, 6, 2, 5, 2, 1680, 'ranked',     0, '2025-10-31'),
    (17, 3, 2, 3, 5, 4, 3240, 'ranked',     0, '2025-11-02'),  -- neonbyte beats zergling #3
    (18, 1, 3, 1, 5, 2, 2520, 'ranked',     0, '2025-11-04'),
    (19, 2, 8, 2, 5, 2, 1980, 'ranked',     0, '2025-11-06'),
    (20, 2, 7, 2, 5, 1, 1560, 'ranked',     0, '2025-11-08'),
    (21, 1, 4, 1, 5, 0, 1440, 'ranked',     0, '2025-11-10'),
    (22, 3, 4, 3, 5, 2, 1920, 'ranked',     0, '2025-11-12'),
    (23, 1, 3, 1, 5, 3, 2760, 'tournament', 0, '2025-11-14'),
    (24, 2, 4, 2, 5, 1, 1620, 'ranked',     0, '2025-11-16'),
    (25, 5, 7, 5, 5, 3, 2340, 'ranked',     0, '2025-11-18'),
    (26, 3, 5, 3, 5, 4, 3060, 'ranked',     0, '2025-11-20');

-- Elo history (K=32 simplified: gap<=50 → ±16, gap>50 fav wins → ±12, gap>50 upset → ±20)
INSERT INTO elo_history VALUES
    ( 1, 1,  1, 1000, 1016,  16),  -- shadow_fox wins M1
    ( 2, 8,  1, 1000,  984, -16),  -- rookie99 loses M1
    ( 3, 2,  2, 1000, 1016,  16),  -- zergling wins M2
    ( 4, 4,  2, 1000,  984, -16),  -- grumpyking loses M2
    ( 5, 3,  3, 1000, 1016,  16),  -- neonbyte wins M3
    ( 6, 6,  3, 1000,  984, -16),  -- pixelwitch loses M3
    ( 7, 5,  4, 1000, 1016,  16),  -- blueshift wins M4
    ( 8, 7,  4, 1000,  984, -16),  -- ironclad loses M4
    ( 9, 1,  5, 1016, 1032,  16),  -- shadow_fox wins M5
    (10, 4,  5,  984,  968, -16),  -- grumpyking loses M5
    (11, 2,  6, 1016, 1032,  16),  -- zergling wins M6
    (12, 6,  6,  984,  968, -16),  -- pixelwitch loses M6
    (13, 3,  7, 1016, 1032,  16),  -- neonbyte wins M7 (H2H #1 vs zergling)
    (14, 2,  7, 1032, 1016, -16),  -- zergling loses M7
    (15, 1,  8, 1032, 1048,  16),  -- shadow_fox wins M8
    (16, 3,  8, 1032, 1016, -16),  -- neonbyte loses M8
    (17, 2,  9, 1016, 1032,  16),  -- zergling wins M9
    (18, 7,  9,  984,  968, -16),  -- ironclad loses M9
    (19, 3, 10, 1016, 1032,  16),  -- neonbyte wins M10
    (20, 4, 10,  968,  952, -16),  -- grumpyking loses M10
    (21, 1, 11, 1048, 1064,  16),  -- shadow_fox wins M11
    (22, 2, 11, 1032, 1016, -16),  -- zergling loses M11
    (23, 3, 12, 1032, 1048,  16),  -- neonbyte wins M12 (H2H #2 vs zergling)
    (24, 2, 12, 1016, 1000, -16),  -- zergling loses M12
    (25, 2, 13, 1000, 1016,  16),  -- zergling wins M13
    (26, 4, 13,  952,  936, -16),  -- grumpyking loses M13
    (27, 1, 14, 1064, 1080,  16),  -- shadow_fox wins M14
    (28, 3, 14, 1048, 1032, -16),  -- neonbyte loses M14
    (29, 5, 15, 1016, 1032,  16),  -- blueshift wins M15
    (30, 6, 15,  968,  952, -16),  -- pixelwitch loses M15
    (31, 2, 16, 1016, 1028,  12),  -- zergling wins M16 (fav, gap=64)
    (32, 6, 16,  952,  940, -12),  -- pixelwitch loses M16
    (33, 3, 17, 1032, 1048,  16),  -- neonbyte wins M17 (H2H #3 vs zergling)
    (34, 2, 17, 1028, 1012, -16),  -- zergling loses M17
    (35, 1, 18, 1080, 1096,  16),  -- shadow_fox wins M18
    (36, 3, 18, 1048, 1032, -16),  -- neonbyte loses M18
    (37, 2, 19, 1012, 1028,  16),  -- zergling wins M19
    (38, 8, 19,  984,  968, -16),  -- rookie99 loses M19
    (39, 2, 20, 1028, 1040,  12),  -- zergling wins M20 (fav, gap=60)
    (40, 7, 20,  968,  956, -12),  -- ironclad loses M20
    (41, 1, 21, 1096, 1108,  12),  -- shadow_fox wins M21 (fav, gap=160)
    (42, 4, 21,  936,  924, -12),  -- grumpyking loses M21
    (43, 3, 22, 1032, 1044,  12),  -- neonbyte wins M22 (fav, gap=108)
    (44, 4, 22,  924,  912, -12),  -- grumpyking loses M22
    (45, 1, 23, 1108, 1120,  12),  -- shadow_fox wins M23 (fav, gap=64)
    (46, 3, 23, 1044, 1032, -12),  -- neonbyte loses M23
    (47, 2, 24, 1040, 1052,  12),  -- zergling wins M24 (fav, gap=128)
    (48, 4, 24,  912,  900, -12),  -- grumpyking loses M24
    (49, 5, 25, 1032, 1044,  12),  -- blueshift wins M25 (fav, gap=76)
    (50, 7, 25,  956,  944, -12),  -- ironclad loses M25
    (51, 3, 26, 1032, 1048,  16),  -- neonbyte wins M26
    (52, 5, 26, 1044, 1028, -16);  -- blueshift loses M26
''')
_conn.commit()

rows = _conn.execute('SELECT username, elo, wins, losses FROM players ORDER BY elo DESC').fetchall()
print(f"{'username':<14} {'elo':>4}  {'W':>2}/{('L')}")
print('-' * 28)
for username, elo, w, l in rows:
    print(f'{username:<14} {elo:>4}  {w:>2}/{l}')


username        elo   W/L
----------------------------
shadow_fox     1120   8/0
zergling       1052   8/4
neonbyte       1048   7/4
blueshift      1028   3/1
rookie99        968   0/2
ironclad        944   0/4
pixelwitch      940   0/4
grumpyking      900   0/7


## 5. Tools

Three tools form a natural reasoning chain:

1. **`get_schema`** — lets the agent discover table structure before writing SQL
2. **`think`** — explicit reasoning slot; the model plans the query before executing it
3. **`run_sql`** — executes a read-only SELECT and returns results as plain text

The system prompt does **not** describe the tables — the agent must call `get_schema` first.

In [5]:
import time


@tool
def get_schema() -> str:
    """Return the database schema — table names and column definitions."""
    rows = _conn.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    return '\n\n'.join(f'-- {name}\n{sql}' for name, sql in rows)


@tool
def think(thought: str) -> str:
    """Scratchpad for reasoning. Write your chain-of-thought here before
    calling run_sql — the thought is captured and returned unchanged."""
    return thought


@tool
def run_sql(query: str) -> str:
    """Execute a read-only SQL SELECT against the game database.
    Returns results as plain text. Only SELECT statements are allowed."""
    if not query.strip().upper().startswith('SELECT'):
        return 'Error: only SELECT statements are permitted.'
    try:
        cur = _conn.execute(query.strip())
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        if not rows:
            return '(no rows returned)'
        header = ' | '.join(cols)
        lines = [header, '-' * len(header)] + [' | '.join(str(v) for v in r) for r in rows]
        return '\n'.join(lines)
    except Exception as e:
        return f'SQL error: {e}'


# Smoke tests
print(get_schema.invoke({})[:120], '...')
print(run_sql.invoke({'query': 'SELECT username, elo FROM players ORDER BY elo DESC LIMIT 3'}))


-- elo_history
CREATE TABLE elo_history (
    id         INTEGER PRIMARY KEY,
    player_id  INTEGER NOT NULL REFERENCES ...
username | elo
--------------
shadow_fox | 1120
zergling | 1052
neonbyte | 1048


## 6. Agent

Standard LangGraph ReAct agent with `MemorySaver` for in-process thread memory.
The system prompt deliberately omits table descriptions — the agent must use
`get_schema` to discover them.

In [6]:
SYSTEM_PROMPT = """You are a helpful analyst for a 1v1 competitive game leaderboard.

You have access to a SQLite database. You do not know the schema upfront — use
get_schema to discover it when needed.

Reasoning discipline:
1. Call get_schema if you are unsure of table or column names.
2. ALWAYS call think with your chain-of-thought before writing any SQL.
3. If a result surprises you, call think again before retrying.
4. Give concise, plain-text answers once you have the data. No markdown.

SQL rules:
- Player IDs are integers. Never filter by username directly on matches or
  elo_history. Always join with the players table:
  e.g. JOIN players p ON p.id = m.player1_id WHERE p.username = 'name'
- Use subqueries or CTEs when a question requires dependent lookups.
"""

checkpointer = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[get_schema, think, run_sql],
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print('Agent ready.')


Agent ready.


## 7. Helper

In [7]:
W         = 300
MAX_LINES = 6   # max lines shown per tool field
_config   = {'configurable': {'thread_id': 'main'}}


def _print_field(label: str, value: str) -> None:
    prefix = f'    {label}: '
    cont   = ' ' * len(prefix)
    avail  = W - len(prefix)
    lines  = str(value).split('\n')

    shown = lines[:MAX_LINES]
    for i, l in enumerate(shown):
        display = l if len(l) <= avail else l[:avail - 1] + '…'
        print((prefix if i == 0 else cont) + display)
    if len(lines) > MAX_LINES:
        print(cont + f'… ({len(lines) - MAX_LINES} more lines)')


def chat(question: str) -> None:
    print(f'\n{"═" * W}')
    print(f' {question}')
    print(f'{"═" * W}')

    pending: dict = {}
    final, final_usage = '', {}

    for chunk in agent.stream(
        {'messages': [{'role': 'user', 'content': question}]},
        config=_config,
        stream_mode='updates',
    ):
        for _, node_output in chunk.items():
            for msg in node_output.get('messages', []):

                if isinstance(msg, AIMessage):
                    agent_usage = msg.usage_metadata or {}
                    if msg.tool_calls:
                        t0 = time.time()
                        for tc in msg.tool_calls:
                            pending[tc['id']] = (tc['name'], tc['args'], t0, agent_usage)
                    elif msg.content:
                        final       = msg.content
                        final_usage = msg.usage_metadata or {}

                elif isinstance(msg, ToolMessage):
                    entry = pending.pop(msg.tool_call_id, None)
                    name, args, t0, agent_u = entry if entry else ('?', {}, time.time(), {})
                    elapsed  = time.time() - t0
                    time_str = f'{elapsed:.2f}s'

                    header = f'  ▸ {name}'
                    print(f'\n{header}{" " * (W - len(header) - len(time_str))}{time_str}')

                    if len(args) == 1:
                        args_str = str(list(args.values())[0])
                    elif args:
                        args_str = ', '.join(f'{k}={v!r}' for k, v in args.items())
                    else:
                        args_str = '(no arguments)'

                    _print_field('in ', args_str)
                    _print_field('out', str(msg.content))
                    if agent_u:
                        _print_field('tok', f"in={agent_u.get('input_tokens','?')}  "
                                            f"out={agent_u.get('output_tokens','?')}")

    if final:
        u = final_usage
        print(f'\n{"─" * W}')
        for l in final.split('\n'):
            display = l if len(l) <= W - 1 else l[:W - 2] + '…'
            print(f' {display}')
        if u:
            print(f'\n tok: in={u.get("input_tokens","?")}  out={u.get("output_tokens","?")}')
        print(f'{"═" * W}')


## 8. Demo runs

### Turn 1 — Leaderboard

Simple query. Expected chain: `get_schema → think → run_sql`.

In [8]:
chat('Show me the leaderboard')


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Show me the leaderboard
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ get_schema                                                                                                                                                                                                                                                                                         0.01s
    in : (no arguments)
    out: -- elo_history
         CREATE TABLE 

### Turn 2 — Follow-up using memory

The agent remembers the schema from Turn 1 and doesn't need to re-call `get_schema`.

In [9]:
chat("What's shadow_fox's win rate?")


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 What's shadow_fox's win rate?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ think                                                                                                                                                                                                                                                                                              0.00s
    in : To calculate shadow_fox's win rate, I need to use the f

### Turn 3 — Join across tables

Requires joining `elo_history` with `matches`. The agent must plan the join via `think`.

In [10]:
chat("What's zergling's Elo history? Show each match date and the rating change.")


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 What's zergling's Elo history? Show each match date and the rating change.
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ run_sql                                                                                                                                                                                                                                                                                            0.01s
    in : SELECT e.m

### Turn 4 — Dependent queries

Requires two SQL calls: find the streak, then look up the starting Elo.
Expected chain: `think → run_sql → think → run_sql`.

In [11]:
chat('Who is on the longest win streak, and what Elo did they start it at?')


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Who is on the longest win streak, and what Elo did they start it at?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ run_sql                                                                                                                                                                                                                                                                                            0.01s
    in : SELECT p.usernam

### Turn 5 — Set up the rivalry question

In [12]:
chat('What is the result of h2h matches between zergling and neonbyte?')


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 What is the result of h2h matches between zergling and neonbyte?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ run_sql                                                                                                                                                                                                                                                                                            0.03s
    in : SELECT m.id, m.winne

### Turn 6 — The rivalry question

zergling sits above neonbyte on the leaderboard, but neonbyte has won every
head-to-head between them. The agent must pull H2H records and Elo side by
side to give a meaningful answer.

In [13]:
chat('Who would win in a match between zergling and neonbyte?')


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Who would win in a match between zergling and neonbyte?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  ▸ think                                                                                                                                                                                                                                                                                              0.00s
    in : Zergling and Neonbyte have fa